In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import time

start_time = time.time()
NUM_RECORDS = 500000
print(f"กำลังสุ่มสร้างข้อมูล OPD ภาษามนุษย์จำนวน {NUM_RECORDS:,} เรคคอร์ด...")

# ==========================================
# 1. โครงสร้างประโยค OPD แบบธรรมชาติ
# ==========================================

# --- คำชม (Positive) ---
pos_intros = ["วันนี้ประทับใจมาก", "บริการดีขึ้นเยอะ", "รวดเร็วทันใจ", "ขอชื่นชมครับ", "ดีเยี่ยมเลย", ""]
pos_subjects = ["คุณหมอ", "พยาบาลจุดซักประวัติ", "เภสัชกรห้องจ่ายยา", "น้องๆ คัดกรอง", "ระบบคิว", "พี่รปภ."]
pos_actions = ["ตรวจละเอียดมาก", "อธิบายวิธีทานยาเข้าใจง่าย", "พูดจาไพเราะยิ้มแย้ม", "จัดคิวได้เป็นระบบดีมาก", "เรียกคิวเร็วมาก", "บริการรวดเร็วไม่ต้องรอนาน"]
pos_details = ["แทบไม่ต้องนั่งรอเลย", "แอร์เย็นที่นั่งรอสบาย", "หาที่จอดรถง่ายขึ้น", "ให้คำแนะนำดีมากๆ", "คุณหมอมือเบามาก"]
pos_endings = ["ขอบคุณมากค่ะ", "รักษามาตรฐานแบบนี้นะครับ", "ยอดเยี่ยมครับ", "ประทับใจสุดๆ", "เป็นกำลังใจให้เจ้าหน้าที่ทุกคนนะคะ", ""]

# --- คำติ/บ่น (Negative) ---
neg_intros = ["ขอติเรื่องนึงนะคะ", "เหนื่อยกับการรอ", "อยากให้ปรับปรุงด่วน", "วันนี้แย่มาก", "แอบผิดหวังระบบ", "ขอระบายหน่อย"]
neg_subjects = ["ระบบคิว", "ที่จอดรถ", "พยาบาลหน้าห้องตรวจ", "ห้องจ่ายยา", "ขั้นตอนทำประวัติ", "ป้ายบอกทาง"]
neg_actions = ["รอนานมหาศาล", "วนหาที่จอดจนท้อ", "จ่ายยาช้ามาก", "โดนแซงคิวเฉยเลย", "พูดจาห้วนมาก", "งงไปหมดไม่รู้ต้องไปช่องไหนต่อ"]
neg_details = ["คิวหน้าจอไม่ตรงกับที่เรียก", "มาตั้งแต่เช้าได้ตรวจบ่าย", "ที่นั่งรอหน้าห้องตรวจไม่พอ", "ถามอะไรก็ชักสีหน้าใส่", "น่าจะจัดระบบให้ดีกว่านี้นะ"]
neg_endings = ["ฝากผู้บริหารจัดการด้วยครับ", "ปรับปรุงด่วนเลยนะคะ", "เสียเวลามาก", "เพลียจิตจริงๆ", "ไม่ไหวจะเคลียร์", ""]

# ==========================================
# 2. ฟังก์ชันเติมความเป็นมนุษย์
# ==========================================
def humanize_text(text, sentiment):
    # สุ่มแปลงคำให้วิบัติ/ลากเสียง (Typo & Slang)
    if random.random() < 0.3:
        text = text.replace("มาก", random.choice(["ม้ากกก", "มากๆ", "ม๊าก"]))
    if random.random() < 0.2:
        text = text.replace("เลย", random.choice(["เลยยย", "เรย", "เลยจ้า"]))
    if random.random() < 0.15:
        text = text.replace("พยาบาล", random.choice(["พยาบาน", "พยบ."]))
        text = text.replace("หมอ", random.choice(["คุมหมอ", "คุณหมอ"]))

    # สุ่มเติมหางเสียง
    if random.random() < 0.5:
        text += random.choice([" ครับ", " ค่ะ", " นะคะ", " ครับผม", " จ้า", " คับ", " แฮะ"])

    # สุ่มใส่อีโมจิ
    if random.random() < 0.4:
        if sentiment == 'positive':
            text += random.choice([" 😊", " ❤️", " 🙏", " 👍", " 🥰", " ✌️"])
        else:
            text += random.choice([" 😡", " 🙄", " 😤", " 👎", " 😩", " 😮‍💨"])

    return text.strip()

def generate_opd_comment(sentiment):
    if sentiment == 'positive':
        components = [random.choice(pos_intros), random.choice(pos_subjects), random.choice(pos_actions), random.choice(pos_details), random.choice(pos_endings)]
    else:
        components = [random.choice(neg_intros), random.choice(neg_subjects), random.choice(neg_actions), random.choice(neg_details), random.choice(neg_endings)]

    raw_text = " ".join([c for c in components if c])
    return humanize_text(raw_text, sentiment)

# ==========================================
# 3. เริ่มสุ่มข้อมูล (รับประกันไม่ซ้ำ)
# ==========================================
data = []
unique_comments = set()
start_date = datetime(2026, 1, 1)

# แผนกใน OPD (อิงจาก รพ. ทั่วไป)
departments = ["อายุรกรรม", "ศัลยกรรม", "กุมารเวชกรรม", "สูตินรีเวช", "ทันตกรรม", "ตา หู คอ จมูก", "กระดูกและข้อ"]

while len(data) < NUM_RECORDS:
    # สุ่มคะแนนภาพรวม (1-5)
    rating = np.random.choice([1, 2, 3, 4, 5], p=[0.08, 0.12, 0.15, 0.35, 0.30])
    sentiment = 'positive' if rating >= 4 else 'negative' if rating <= 2 else random.choice(['positive', 'negative'])

    comment = generate_opd_comment(sentiment)

    # กันข้อมูลซ้ำ 100%
    while comment in unique_comments:
        comment += random.choice([" ", "!", ".", "คะ", "ครับ", "~"])
    unique_comments.add(comment)

    # สุ่มคะแนนรายข้อ 9 ข้อ (อิงจากภาพฟอร์ม OPD) - ให้คะแนนล้อไปกับ Overall Rating
    if sentiment == 'positive':
        q_scores = np.random.choice([4, 5], 9, p=[0.4, 0.6])
    elif sentiment == 'negative':
        q_scores = np.random.choice([1, 2, 3], 9, p=[0.4, 0.4, 0.2])
    else:
        q_scores = np.random.choice([3, 4], 9, p=[0.5, 0.5])

    # สุ่มวันเวลา
    ts = start_date + timedelta(days=random.randint(0, 90), hours=random.randint(8, 15), minutes=random.randint(0, 59))

    data.append({
        'feedback_id': f'F_OPD_{str(len(data)+1).zfill(5)}',
        'timestamp': ts.strftime('%Y-%m-%d %H:%M'),
        'department': random.choice(departments),
        'q1_doctor': q_scores[0],
        'q2_nurse': q_scores[1],
        'q3_queue': q_scores[2],
        'q4_staff': q_scores[3],
        'q5_speed': q_scores[4],
        'q6_cleanliness': q_scores[5],
        'q7_signage': q_scores[6],
        'q8_parking': q_scores[7],
        'q9_overall_rating': rating,
        'feedback_text': comment
    })

# ==========================================
# 4. บันทึกไฟล์
# ==========================================
df = pd.DataFrame(data)

print("\n--- ตัวอย่างข้อมูล OPD แบบภาษามนุษย์ (10 แถวแรก) ---")
for index, row in df.head(10).iterrows():
    print(f"[{row['department']}] Rating:{row['q9_overall_rating']} -> {row['feedback_text']}")

file_name = 'suth_hospital_opd_natural_20k.csv'
df.to_csv(file_name, index=False, encoding='utf-8-sig')

process_time = time.time() - start_time
print(f"\n✅ สร้างข้อมูล OPD สำเร็จ! {len(df):,} เรคคอร์ด")
print(f"📁 บันทึกไฟล์ '{file_name}' เรียบร้อย ใช้เวลาไป {process_time:.2f} วินาที")

กำลังสุ่มสร้างข้อมูล OPD ภาษามนุษย์จำนวน 500,000 เรคคอร์ด...

--- ตัวอย่างข้อมูล OPD แบบภาษามนุษย์ (10 แถวแรก) ---
[ตา หู คอ จมูก] Rating:2 -> ขอติเรื่องนึงนะคะ พยาบาลหน้าห้องตรวจ งงไปหมดไม่รู้ต้องไปช่องไหนต่อ ถามอะไรก็ชักสีหน้าใส่ ไม่ไหวจะเคลียร์ จ้า
[อายุรกรรม] Rating:2 -> ขอระบายหน่อย พยบ.หน้าห้องตรวจ รอนานมหาศาล ถามอะไรก็ชักสีหน้าใส่
[อายุรกรรม] Rating:4 -> เภสัชกรห้องจ่ายยา อธิบายวิธีทานยาเข้าใจง่าย คุณหมอมือเบามาก ครับ 😊
[ตา หู คอ จมูก] Rating:4 -> รวดเร็วทันใจ คุณหมอ อธิบายวิธีทานยาเข้าใจง่าย คุณหมอมือเบามาก รักษามาตรฐานแบบนี้นะครับ คับ
[กระดูกและข้อ] Rating:3 -> ดีเยี่ยมเลย น้องๆ คัดกรอง ตรวจละเอียดมาก แทบไม่ต้องนั่งรอเลย รักษามาตรฐานแบบนี้นะครับ
[ทันตกรรม] Rating:2 -> วันนี้แย่มาก ขั้นตอนทำประวัติ จ่ายยาช้ามาก น่าจะจัดระบบให้ดีกว่านี้นะ เสียเวลามาก
[อายุรกรรม] Rating:1 -> แอบผิดหวังระบบ ขั้นตอนทำประวัติ โดนแซงคิวเฉยเลยยย ที่นั่งรอหน้าห้องตรวจไม่พอ เสียเวลามาก
[อายุรกรรม] Rating:4 -> รวดเร็วทันใจ ระบบคิว จัดคิวได้เป็นระบบดีมาก คุณหมอมือเบามาก ประทับใจสุดๆ นะคะ ✌️
[กระดูกและข้อ]